## 11. Optional: Hyperparameter Tuning

Tuning tries multiple parameter combinations with `TimeSeriesSplit`. This can take a long time. Use it only after the simple training run works.


In [1]:
from pathlib import Path
import os
os.environ["OMP_NUM_THREADS"] = "48"
import sys
import importlib
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

# Make local imports work when the notebook is opened from another folder.
PIPELINE_DIR = Path.cwd()
if not (PIPELINE_DIR / 'loader.py').exists():
    PIPELINE_DIR = Path.cwd() / 'pipeline'

if str(PIPELINE_DIR) not in sys.path:
    sys.path.append(str(PIPELINE_DIR))

from loader import LoaderStorage
from targets import add_delay_class_target, DELAY_CLASS_ORDER
from split import chronological_train_val_test_split
from features import  check_columns
from evaluate import plotConfusionMatrix
import train
importlib.reload(train)
from train import TrainingConfig, run_training
import models
importlib.reload(models)


<module 'models' from 'c:\\Users\\simon schumacher\\OneDrive\\Desktop\\FSS_26_Master_1\\DataMining\\Project\\Proj\\ie500_data_mining_project\\pipeline\\models.py'>

In [ ]:
# Change these paths to your actual dataset location.
DATA_ROOT = "s3://data-mining"
INPUT_PATH = 'data/features/n3_feature_engineered.parquet'

# Column names used by the current pipeline.
DELAY_COLUMN = 'ArrDelayMinutes'
TIME_COLUMN = 'CRSDepDateTime_UTC'
TARGET_COLUMN = 'delay_class'

# Use a small sample while learning/debugging. Set to 1.0 for the final run.
SAMPLE_FRAC = 0.01

# Good first choices: 'dummy', 'logistic_regression', 'random_forest', 'hist_gradient_boosting', 'xgboost', 'svc'.
MODEL_NAMES = ['naive_bayes'] #'svc']

OUTPUT_DIR = 'outputs/training_notebook'


In [3]:
%%time
%time
# Uncomment this cell when you are ready for a slower tuning run.
for n in MODEL_NAMES:
  print(f"TRAINING {n}")
  tuned_config = TrainingConfig(
      data_root=DATA_ROOT,
      input_path=INPUT_PATH,
      output_dir='outputs/training_notebook_tuned',
      model_name=n,
      delay_column=DELAY_COLUMN,
      target_column=TARGET_COLUMN,
      time_column=TIME_COLUMN,
      sample_frac=SAMPLE_FRAC,
      weights=None, # ignored during tuning; class weights are sampled from intervals in RandomizedSearchCV.
      tune=True,
      n_iter=2,
      cv_splits=3,
  )
  tuned_metrics,best_params = run_training(tuned_config)
  display(pd.Series(tuned_metrics, name='tuned_pipeline_metrics'))
  %time
  if best_params:
      print("Best hyperparameters found during tuning:")
      for param, value in best_params.items():
          print(f"{param}: {value}")

CPU times: total: 0 ns
Wall time: 0 ns
TRAINING naive_bayes
load dataframe complete
CPU times: total: 2min 2s
Wall time: 8min 34s


MemoryError: Unable to allocate 6.24 GiB for an array with shape (61, 13740088) and data type float64